# MassGAT: End-to-End Inference Pipeline
This notebook demonstrates the complete MassGAT workflow on a single LC-MS file. The process is divided into three stages:
1. **IonShed (R):** Raw data segmentation and ROI extraction.
2. **Graph Construction (Python):** Building `.pt` chemical graphs.
3. **Inference (Python):** Pruning of candidate chemical components using MassGAT

Additionally, we provide a visualization tool to explore the resulting chemical components.

In [1]:
import os
from pathlib import Path

# Define the project workspace and create necessary directories
os.environ["PROJ_NAME"] = "test"
DOC_DIR = Path(os.getcwd())
BASE_DIR = DOC_DIR.parent
DATA_DIR = BASE_DIR / "data" / os.environ["PROJ_NAME"]
MZML_DIR = DATA_DIR / "mzml"
IONSHED_DIR = DATA_DIR / "ionshed_output"
GRAPHS_DIR = DATA_DIR / "graphs"
RESULTS_DIR = DATA_DIR / "results"

# Create directories
for d in [MZML_DIR, IONSHED_DIR, GRAPHS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Define model paths and parameters
WEIGHTS_PATH = BASE_DIR / "weights" / "model_positive.pth"
IONIZATION_MODE = "positive" # "positive" or "negative"

print(f"Please place your test .mzML file in: {MZML_DIR}")
print(f"Place positive and negative files in separate folder and select the correct ionization mode in this cell.")

Please place your test .mzML file in: c:\Users\SD265344\Documents\006_Packages\massgat\data\test\mzml
Place positive and negative files in separate folder and select the correct ionization mode in this cell.


---
## Stage 1: IonShed Processing (R)
We use the `rpy2` bridge to execute native R code within this Python notebook. This step reads the `.mzML` file, performs descending peak picking, extracts XICs, and saves the outputs as `.feather` files.

In [2]:
import os
import logging
import warnings

os.environ["RPY2_CFFI_MODE"] = "ABI"
warnings.filterwarnings('ignore', category=UserWarning, module='rpy2')
warnings.filterwarnings('ignore', category=ImportWarning, module='rpy2')

# Load the rpy2 extension (for R integration in Jupyter notebooks)
%load_ext rpy2.ipython

import rpy2.rinterface_lib.callbacks
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Convert Windows paths to simple strings for R compatibility
MZML_DIR_STR = MZML_DIR.as_posix()
IONSHED_DIR_STR = IONSHED_DIR.as_posix()

In [3]:
%%R -i MZML_DIR_STR -i IONSHED_DIR_STR

# Load required packages
Sys.setlocale("LC_ALL", "C")
invisible(suppressPackageStartupMessages({
    library(ionshed)
    library(arrow)
    library(xcms)
    library(MSnbase)
}))

mzml_files <- list.files(MZML_DIR_STR, pattern = "\\.(mzML|mzXML)$", full.names = TRUE, ignore.case = TRUE)

if (length(mzml_files) == 0) {
    stop("No .mzML files found. Please add a file to the mzml directory.")
}

file_path <- mzml_files[1]
base_name <- sub("\\.(mzML|mzXML)$", "", basename(file_path), ignore.case = TRUE)
peak_output <- file.path(IONSHED_DIR_STR, paste0(base_name, "_peak.feather"))
roi_output <- file.path(IONSHED_DIR_STR, paste0(base_name, "_signal.feather"))

cat("Processing:", base_name, "...\n")

# Load the MS data and perform peak picking and ROI extraction
MSnExpObj <- suppressMessages(readMSData(file_path, mode = "onDisk", msLevel. = 1L))
res <- suppressMessages(ionshed(MSnExpObj, rttol = 5, ppm = 3, mz_res = 0.0001, rt_res = 0.1))
roi_res <- suppressMessages(extract_xics(roi_ls = res, rttol = 10))

write_feather(res$peak_info, peak_output)
write_feather(res$roi_data, roi_output)

cat("IonShed processing complete. Files saved successfully.\n")

Processing: positive ...
IonShed processing complete. Files saved successfully.


In addition: There were 32 warnings (use warnings() to see them)


---
## Stage 2: Hierarchical Graph Construction
We now convert the extracted `.feather` tables into PyTorch Geometric graphs (`.pt`), defining nodes (ion traces) and edges (chemical relationships). We call your standalone script directly to mirror the HPC execution.

In [9]:
import subprocess

print("Building graphs from IonShed outputs...")

# Execute the build_graphs.py script
cmd_build = [
    "python", "../scripts/build_graphs.py",
    "--input_dir", str(IONSHED_DIR),
    "--output_dir", str(GRAPHS_DIR),
    "--ionization_mode", IONIZATION_MODE
]

result = subprocess.run(cmd_build, capture_output=True, text=True)

if result.returncode == 0:
    print("Graph construction successful.")
    print(result.stdout.split("\n")[-3])
else:
    print("Error during graph construction:")
    print(result.stderr)

Building graphs from IonShed outputs...
Graph construction successful.
[2026-07-28 14:25:33] General processing completed


---
## Stage 3: MassGAT Inference
Finally, we run MassGAT on the generated graphs. This step evaluates graph probabilities, node probabilities, and outputs the final `.tsv` tables containing validated annotated components.

In [10]:
print("Running MassGAT inference...")

# Execute the predict.py script
cmd_predict = [
    "python", "../scripts/predict.py",
    "--folder", str(GRAPHS_DIR),
    "--data_type", "real",
    "--weights", str(WEIGHTS_PATH),
    "--model_type", "iterative",
    "--encoder_type", "transformer",
    "--hidden_dim", "32",
    "--embedding_dim", "64",
    "--block_type", "gate",
    "--min_node_prob", "0.5", # set to 0.0 to have all the segments in the output(not recommended)
    "--output_dir", str(RESULTS_DIR),
    "--ionshed_output_dir", str(IONSHED_DIR), # Only needed for xcms alignment
    "--with_top_annotations" # If you want suggested annotations byt the heuristic in the peak table
]

result = subprocess.run(cmd_predict, capture_output=True, text=True)

if result.returncode == 0:
    print("Inference successful. Check the results directory for TSV outputs.")
    # Print the last few lines of the output for the summary
    for line in result.stdout.split('\n')[-6:]:
        if line.strip():
            print(line)
else:
    print("Error during inference:")
    print(result.stderr)

Running MassGAT inference...
Error during inference:
Traceback (most recent call last):
  File "c:\Users\SD265344\Documents\006_Packages\massgat\scripts\predict.py", line 15, in <module>
    from src.utils.annotate_component import annotate_component
  File "C:\Users\SD265344\Documents\006_Packages\massgat\src\utils\__init__.py", line 4, in <module>
    from .mlflow_utils import (setup_mlflow, log_training_params, log_training_metrics,
  File "C:\Users\SD265344\Documents\006_Packages\massgat\src\utils\mlflow_utils.py", line 4, in <module>
    import mlflow
  File "c:\Users\SD265344\Documents\006_Packages\massgat\.pixi\envs\cpu\lib\site-packages\mlflow\__init__.py", line 32, in <module>
    import mlflow.tracking._model_registry.fluent
  File "c:\Users\SD265344\Documents\006_Packages\massgat\.pixi\envs\cpu\lib\site-packages\mlflow\tracking\__init__.py", line 8, in <module>
    from mlflow.tracking.client import MlflowClient
  File "c:\Users\SD265344\Documents\006_Packages\massgat\.pixi\

---
## Stage 4: Results visualization
Use the interface below to search for a specific chemical target in your processed graphs. Enter your expected **m/z** and **Retention Time (min)** to locate the component, extract the pseudo-spectrum, and visualize MassGAT's predictions.

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd, torch, networkx as nx, glob
from pathlib import Path

from src.utils.visualizations import create_nx_graph, visualize_component

# Setup directories for visualizations
VIS_DIR = DATA_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

display(HTML("<style>.search-card{background:#fff;border:1px solid #e5e7eb;border-radius:12px;padding:16px;box-shadow:0 1px 3px rgba(0,0,0,.06);} .search-card .widget-dropdown>select,.search-card .widget-text>input,.search-card .widget-numeric-text>input{border-radius:8px!important;} .search-card .widget-button{border-radius:8px!important;}</style>"))

ui_style = {'description_width': 'initial'}
tsv_files = [Path(p).name for p in glob.glob(str(RESULTS_DIR / "*_metadata.tsv"))]

w_file = widgets.Dropdown(options=tsv_files, description='Results TSV:', style=ui_style)
w_graph = widgets.Dropdown(options=[], description='Graph (.pt):', style=ui_style)
w_mz = widgets.FloatText(value=152.0556, description='Target m/z:', step=0.0001, style=ui_style)
w_rt = widgets.FloatText(value=4.47, description='Target RT (min):', step=0.01, style=ui_style)
w_ppm = widgets.FloatText(value=5.0, description='Tol (ppm):', step=1.0, style=ui_style)
w_title = widgets.Text(value='Guanine', description='Save Title:', style=ui_style)

w_btn = widgets.Button(description='Search', button_style='primary', icon='search')
w_save = widgets.Button(description='Save HTML', button_style='success', icon='save', disabled=True)
w_match = widgets.Dropdown(options=[], description='Select Match:', style=ui_style, disabled=True)

out_plot = widgets.Output()

# Global state to cache data and avoid reloading files when switching matches
state = {'graph': None, 'G': None, 'node_probs': None}
last_args = {}

# Update graph options based on selected TSV file
def update_graph_options(*a):
    if not w_file.value: return w_graph.options.clear()
    try: w_graph.options = pd.read_csv(RESULTS_DIR / w_file.value, sep='\t', usecols=['file_name'])['file_name'].dropna().unique().tolist()
    except: w_graph.options = []

w_file.observe(update_graph_options, names='value')
update_graph_options()

# Render logic: triggers when a specific match is selected
def render_selected_node(change):
    if change['type'] != 'change' or change['name'] != 'value' or change['new'] is None: return
    
    node = change['new']
    w_save.disabled = True
    with out_plot:
        clear_output(wait=True)
        component = nx.node_connected_component(state['G'], node)
        
        print(f"✅ Rendering Node {node} (Component size: {len(component)} peaks)")
        last_args.update(dict(G=state['G'], nodes=component, highlight_nodes=[node], node_probs=state['node_probs'], 
                              graph_name=w_graph.value, ionization_mode=IONIZATION_MODE, remove_low_prob=True,
                              use_prob_styling=True, show_profiles=True, 
                              graph=state['graph'], style="presentation"))
        
        visualize_component(**last_args, show=True)
        w_save.disabled = False

w_match.observe(render_selected_node)

# Search logic: Loads files and finds all matches
def on_search(b):
    w_save.disabled = True
    w_match.disabled = True
    w_match.options = []
    
    with out_plot:
        clear_output(wait=True)
        if not w_file.value or not w_graph.value: return print("Select TSV and Graph.")
        
        print("Scanning files...")
        df = pd.read_csv(RESULTS_DIR / w_file.value, sep='\t')
        state['node_probs'] = {(r['file_name'], r['roi_id']): r['node_prob'] for _, r in df.iterrows()}
        state['graph'] = torch.load(GRAPHS_DIR / w_graph.value, map_location='cpu')
        state['G'] = create_nx_graph(state['graph'], ionization_mode=IONIZATION_MODE)
        
        mz_feat, rt_feat_sec = state['graph']["node_features"][:, 0], state['graph']["node_features"][:, 1]
        matches = torch.where((torch.abs(mz_feat - w_mz.value) <= (w_mz.value * w_ppm.value / 1e6)) & 
                              (torch.abs(rt_feat_sec - (w_rt.value * 60.0)) <= 5.0))[0]
        
        if not len(matches): return print("❌ No matches found. Try widening your tolerances.")
        
        # Build dropdown options
        options = []
        for m in matches:
            idx = m.item()
            m_mz = mz_feat[idx].item()
            m_rt = rt_feat_sec[idx].item() / 60.0
            options.append((f"Node {idx} (m/z: {m_mz:.4f}, RT: {m_rt:.2f}m)", idx))
        
        print(f"🎯 Found {len(options)} match(es).")
        
        w_match.options = options
        w_match.disabled = False

def on_save(b):
    with out_plot:
        base = w_graph.value.replace("graph_", "").replace(".pt", "")
        node_id = w_match.value
        path = VIS_DIR / f"{base}_{w_title.value}_{w_mz.value}_{w_rt.value}_node{node_id}.html"
        visualize_component(**last_args, save_html=str(path), show=False)
        print(f"💾 Saved to: {path}")

w_btn.on_click(on_search)
w_save.on_click(on_save)

ui = widgets.VBox([
    widgets.HBox([w_file, w_graph]), 
    widgets.HBox([w_mz, w_rt, w_ppm]), 
    widgets.HBox([w_btn, w_match]),
    widgets.HBox([w_title, w_save])
])
ui.add_class('search-card')
display(ui, out_plot)

ImportError: cannot import name 'service' from 'google.protobuf' (c:\Users\SD265344\Documents\006_Packages\massgat\.pixi\envs\cpu\lib\site-packages\google\protobuf\__init__.py)